# 2.4 — DEP-RL baseline (myoLegWalk)

Load the pretrained [DEP-RL](https://github.com/martius-lab/depRL) walk policy and roll it out on `myoLegWalk-v0`.

**Requirements**
- Python **≥3.9 and ≤3.11.5** (`deprl` on PyPI does not support 3.12+)
- `pip install deprl`
- MyoSuite installed from this repo (`pip install -e .`)

Skip this notebook if you do not need the DEP-RL baseline; prefer [2.1 — SB3](./2.1_Train_SB3_Policy.ipynb) for general RL training.

In [ ]:
import sys

try:
    import deprl  # type: ignore
    from deprl import env_wrappers  # type: ignore

    DEPRL_AVAILABLE = True
except ImportError:
    DEPRL_AVAILABLE = False
    print(
        "deprl is not installed; skipping DEP-RL cells.\n"
        "Install with: pip install deprl\n"
        f"(requires Python ≤3.11.5; this interpreter is {sys.version.split()[0]})"
    )

Create the walk env, wrap it for DEP-RL, load the published baseline, and run one episode.

Notes:
- `GymWrapper.reset()` returns a bare observation array (not `(obs, info)`).
- The baseline emits actions in `[-1, 1]`; current `myoLegWalk-v0` expects muscle activations in `[0, 1]`, so we map with `0.5 * (a + 1)`.

In [ ]:
if DEPRL_AVAILABLE:
    import numpy as np
    import myosuite  # noqa: F401 — register envs
    from myosuite.utils import gym
    import deprl
    from deprl import env_wrappers

    T = 1000
    env = gym.make("myoLegWalk-v0")
    env = env_wrappers.GymWrapper(env)

    policy = deprl.load_baseline(env)

    obs = env.reset()
    if isinstance(obs, tuple):
        obs = obs[0]

    total_reward = 0.0
    for step in range(T):
        action = np.asarray(policy(obs), dtype=np.float32)
        action = np.clip(0.5 * (action + 1.0), 0.0, 1.0)
        obs, rew, done, *rest = env.step(action)
        total_reward += float(rew)
        if done:
            print(f"Episode ended at step {step + 1}")
            break
    else:
        print(f"Ran full horizon ({T} steps)")

    env.close()
    print(f"Done! return = {total_reward:.2f}")

To load your own checkpoint instead of the published baseline, use `deprl.load(path, env)`. For a viewer script: `python -m deprl.play --path /folder/`.